In [ ]:
# Basic imports for transformer-based text classification
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# Load preprocessed and augmented data from 01_preprocessing.ipynb
%run 01_preprocessing.ipynb

# Display info about the loaded augmented data
print(f"\n{'='*60}")
print(f"Loaded train_df_clean (augmented): {len(train_df_clean)} rows")
print(f"Columns: {list(train_df_clean.columns)}")
print(f"\nLabel distribution:")
print(train_df_clean['rule_violation'].value_counts())
print(f"{'='*60}\n")

In [ ]:
# Configuration for transformer training
class CFG:
    model_name_or_path = "distilbert-base-uncased"  # Options: "bert-base-uncased", "roberta-base", "microsoft/deberta-v3-base"
    output_dir = "./transformer_model"
    
    EPOCHS = 3
    LEARNING_RATE = 2e-5
    MAX_LENGTH = 512
    BATCH_SIZE = 8
    RANDOM_SEED = 42

print("✓ Configuration set!")
print(f"  Model: {CFG.model_name_or_path}")
print(f"  Epochs: {CFG.EPOCHS}, LR: {CFG.LEARNING_RATE}, Batch: {CFG.BATCH_SIZE}")

In [ ]:
# Custom Dataset class for transformers
class RedditRulesDataset(torch.utils.data.Dataset):
    """Custom PyTorch Dataset for Reddit rules classification"""
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

print("✓ Custom Dataset class defined!")

In [ ]:
# Initialize tokenizer
print(f"Loading tokenizer: {CFG.model_name_or_path}")
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name_or_path)

print("✓ Tokenizer loaded!")

In [ ]:
# Tokenize training data (using the combined_text column from preprocessing)
print("Tokenizing training data...")
train_texts = train_df_clean['combined_text'].tolist()
train_labels = train_df_clean['rule_violation'].tolist()

train_encodings = tokenizer(
    train_texts, 
    truncation=True, 
    padding=True, 
    max_length=CFG.MAX_LENGTH
)

# Create dataset
train_dataset = RedditRulesDataset(train_encodings, train_labels)

print(f"✓ Tokenization complete!")
print(f"  Dataset size: {len(train_dataset)}")
print(f"  Label distribution: {np.bincount(train_labels)}")

In [ ]:
# Load pre-trained model for sequence classification
print(f"Loading model: {CFG.model_name_or_path}")
model = AutoModelForSequenceClassification.from_pretrained(
    CFG.model_name_or_path, 
    num_labels=2  # Binary classification: violation (1) or no violation (0)
)

# Move model to device
model = model.to(device)

print(f"✓ Model loaded!")
print(f"  Model type: {type(model).__name__}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=CFG.output_dir,
    num_train_epochs=CFG.EPOCHS,
    learning_rate=CFG.LEARNING_RATE,
    per_device_train_batch_size=CFG.BATCH_SIZE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="no",
    report_to="none",
    seed=CFG.RANDOM_SEED,
)

print("✓ Training arguments configured")

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("✓ Trainer initialized")

In [ ]:
# Train the model
print("\n" + "="*60)
print("Starting training...")
print("="*60 + "\n")

trainer.train()

print("\n" + "="*60)
print("✓ Training complete!")
print(f"Model saved to: {CFG.output_dir}")
print("="*60)

In [ ]:
# Load and preprocess test data
import sys
sys.path.append("../src")
from preprocessing import preprocess_dataframe, combine_comment_rule

test_df = pd.read_csv("../data/raw/test.csv")
print(f"Test dataset size: {len(test_df)}")

# Preprocess test data the same way as training data
columns_to_clean = ["body", "rule"]
test_df_clean = preprocess_dataframe(test_df, columns_to_clean, label_column=None)
test_df_clean = combine_comment_rule(test_df_clean, "body", "rule", "combined_text")

print(f"✓ Test data preprocessed")
test_df_clean.head()

In [ ]:
# Tokenize test data
print("Tokenizing test data...")
test_texts = test_df_clean['combined_text'].tolist()

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=CFG.MAX_LENGTH
)

test_dataset = RedditRulesDataset(test_encodings, labels=None)

print(f"✓ Test data tokenized: {len(test_dataset)} samples")

In [ ]:
# Make predictions on test set
print("Making predictions...")
predictions = trainer.predict(test_dataset)

# Get probability scores for class 1 (rule violation)
probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

print(f"✓ Predictions complete!")
print(f"  Min probability: {probs.min():.4f}")
print(f"  Max probability: {probs.max():.4f}")
print(f"  Mean probability: {probs.mean():.4f}")

In [ ]:
# Create submission file
submission_df = pd.DataFrame({
    "row_id": test_df["row_id"],
    "rule_violation": probs
})

submission_filename = "submission_transformer.csv"
submission_df.to_csv(submission_filename, index=False)

print(f"✓ Submission file created: {submission_filename}")
print(f"\nSubmission statistics:")
print(f"  Total predictions: {len(submission_df)}")
print(f"  Predictions >= 0.5: {(probs >= 0.5).sum()} ({(probs >= 0.5).sum()/len(probs)*100:.1f}%)")
print(f"  Predictions < 0.5: {(probs < 0.5).sum()} ({(probs < 0.5).sum()/len(probs)*100:.1f}%)")

print(f"\nFirst 10 rows:")
submission_df.head(10)

In [ ]:
# Training complete! The augmented dataset from preprocessing was used.
# This includes:
# - Original body rows from train.csv
# - All positive/negative examples from train.csv as new rows
# - All positive/negative examples from test.csv as new rows
# - Example columns removed, only core columns remain
# Total rows are significantly more than the original dataset